# C02 财务 + 衍生 + 宏观因子

本节目标：把多源数据统一成 `factor_panel`，供后续回测使用。


## 三类因子来源
- 财务因子（慢频）
- 技术因子（快频）
- 宏观因子（全市场时间序列）

核心是“对齐到同一时间轴”。


In [ ]:
MODE = "offline"
START_DATE = "2020-01-01"
END_DATE = "2021-12-31"
UNIVERSE = ["000001.XSHE", "000002.XSHE", "600000.XSHG", "601318.XSHG"]
SEED = 42

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(SEED)


def resolve_mode(mode):
    # 统一入口：优先尝试 online，失败后自动回退 offline
    if mode != "online":
        return "offline", None
    try:
        import rqdatac
        try:
            rqdatac.init()
        except Exception:
            pass
        return "online", rqdatac
    except Exception as e:
        print(f"[INFO] online 不可用，自动切换到 offline: {e}")
        return "offline", None

DATA_MODE, rqdatac = resolve_mode(MODE)
print("DATA_MODE:", DATA_MODE)


## 财务口径提示
- `mrq`：最近报告期
- `ttm`：最近四季滚动
- `lyr`：上一个完整财年

课堂讲解重点：避免未来函数（发布时间对齐）。


In [ ]:
def make_financials(universe, start_date, end_date):
    quarters = pd.period_range(start=start_date, end=end_date, freq="Q")
    rows = []
    for asset in universe:
        rev0 = np.random.uniform(50, 260)
        p0 = np.random.uniform(5, 40)
        g = np.random.uniform(-0.01, 0.06)
        for i, q in enumerate(quarters):
            rev = rev0 * (1 + g) ** i * np.random.uniform(0.9, 1.1)
            pft = p0 * (1 + g) ** i * np.random.uniform(0.85, 1.15)
            rows.append({
                "date": q.end_time,
                "order_book_id": asset,
                "revenue_mrq": rev,
                "net_profit_mrq": pft,
            })
    fin = pd.DataFrame(rows).sort_values(["order_book_id", "date"])
    fin["revenue_ttm"] = fin.groupby("order_book_id")["revenue_mrq"].rolling(4).sum().reset_index(level=0, drop=True)
    fin["net_profit_ttm"] = fin.groupby("order_book_id")["net_profit_mrq"].rolling(4).sum().reset_index(level=0, drop=True)
    fin["net_margin_ttm"] = fin["net_profit_ttm"] / fin["revenue_ttm"]
    return fin

financial_df = make_financials(UNIVERSE, START_DATE, END_DATE)
financial_df.tail(8)


## 衍生与宏观因子示例
本节保留最小可讲组合：
- `mom20`：20 日动量
- `vol20`：20 日波动率
- `m2_yoy`、`cn10y`：宏观代理序列


In [ ]:
def make_prices(universe, start_date, end_date):
    dates = pd.bdate_range(start_date, end_date)
    panel = {}
    for i, asset in enumerate(universe):
        ret = np.random.normal(0.0003 + i * 0.00005, 0.012, size=len(dates))
        panel[asset] = 30 * np.cumprod(1 + ret)
    return pd.DataFrame(panel, index=dates)

price = make_prices(UNIVERSE, START_DATE, END_DATE)
mom20 = price.pct_change(20)
vol20 = price.pct_change().rolling(20).std()

macro_dates = pd.date_range(START_DATE, END_DATE, freq="ME")
macro = pd.DataFrame({
    "date": macro_dates,
    "m2_yoy": np.random.normal(0.08, 0.01, len(macro_dates)),
    "cn10y": np.random.normal(0.028, 0.002, len(macro_dates)),
})

macro.tail()


## 对齐与标准化（重点）
- 全部映射到月频截面
- 按日期做横截面 z-score


In [ ]:
rows = []
for d in pd.date_range(START_DATE, END_DATE, freq="ME"):
    d_biz = price.index[price.index <= d].max()
    if pd.isna(d_biz):
        continue
    for asset in UNIVERSE:
        rows.append({
            "date": d_biz,
            "order_book_id": asset,
            "mom20": mom20.loc[d_biz, asset],
            "vol20": vol20.loc[d_biz, asset],
        })

factor_panel = pd.DataFrame(rows).merge(macro, on="date", how="left")
for col in ["mom20", "vol20"]:
    factor_panel[col] = factor_panel.groupby("date")[col].transform(lambda x: (x - x.mean()) / (x.std() + 1e-8))

factor_panel = factor_panel.sort_values(["date", "order_book_id"]).reset_index(drop=True)
factor_panel.head(12)


In [ ]:
# 快速检查缺失率，确认面板质量
na_report = factor_panel[["mom20", "vol20", "m2_yoy", "cn10y"]].isna().mean().rename("na_ratio")
na_report


## 小结与练习
- 本节产出：`factor_panel`
- 练习：新增一个 size proxy 因子，并接入标准化流程。
